# Code for the training of the $C_{64}$-equivariant NN - 8.1  Architecture for **$2\pi/(64)=\pi/(32)$ rotations**:

## Input(250^2) -> W1(100) -> BN -> GELU -> Drop -> W2(64) -> BN -> GELU -> Drop -> W3(32) -> BN -> Sigma2 -> Output(2)


## Applies dropout once only and uses Historical statistics for OrbitBN 

## Modify to fit $\pi/(32)$-equivariance:

- 4 layers
    ```python
  Input (250^2)
    → W1 (100 x 250^2)      # W_1 G = phi_1 W_1 and phi_1 is the 64 x 64 permutation, G loaded from MTX_FILENAME = "../Files/BetterACpermutation_matrix_250_5.625.mtx" and G_10 =  MTX_FILENAME = "../Files/BetterACpermutation_matrix_5.625.mtx" (W1 is 100 x 250^2) 
    → OrbitBatchNorm (G)
    → GELU
    → OrbitDropout (G)
    → W2 (64 x 100)        # W_2 G_10 = phi_2 W_2, phi_2 is the a 64 x 64 permutation (W2 is 64 x 100)
    → OrbitBatchNorm (Φ)
    → GELU 
    → OrbitDropout (G)
    → W3 (32 x 64)            # W_3 phi_2 = phi_3 W_3, phi_3 is  32 X 32 permutation
    → OrbitBatchNorm (Φ)   
    → Sigma2 (L + norm-based) # Matrix L form below
    → Output (2)
  ```

####  Symbolic Matrix for K=64 
```python
Row 1 (Cosines):

['cos(0)', 'cos(pi/16)', 'cos(pi/8)', 'cos(3pi/16)', 'cos(pi/4)', 'cos(5pi/16)', 'cos(3pi/8)', 'cos(7pi/16)', 'cos(pi/2)', 'cos(9pi/16)', 'cos(5pi/8)', 'cos(11pi/16)', 'cos(3pi/4)', 'cos(13pi/16)', 'cos(7pi/8)', 'cos(15pi/16)', 'cos(pi)', 'cos(17pi/16)', 'cos(9pi/8)', 'cos(19pi/16)', 'cos(5pi/4)', 'cos(21pi/16)', 'cos(11pi/8)', 'cos(23pi/16)', 'cos(3pi/2)', 'cos(25pi/16)', 'cos(13pi/8)', 'cos(27pi/16)', 'cos(7pi/4)', 'cos(29pi/16)', 'cos(15pi/8)', 'cos(31pi/16)']

Row 2 (Sines):
['sin(0)', 'sin(pi/16)', 'sin(pi/8)', 'sin(3pi/16)', 'sin(pi/4)', 'sin(5pi/16)', 'sin(3pi/8)', 'sin(7pi/16)', 'sin(pi/2)', 'sin(9pi/16)', 'sin(5pi/8)', 'sin(11pi/16)', 'sin(3pi/4)', 'sin(13pi/16)', 'sin(7pi/8)', 'sin(15pi/16)', 'sin(pi)', 'sin(17pi/16)', 'sin(9pi/8)', 'sin(19pi/16)', 'sin(5pi/4)', 'sin(21pi/16)', 'sin(11pi/8)', 'sin(23pi/16)', 'sin(3pi/2)', 'sin(25pi/16)', 'sin(13pi/8)', 'sin(27pi/16)', 'sin(7pi/4)', 'sin(29pi/16)', 'sin(15pi/8)', 'sin(31pi/16)']

File names:

MTX_FILENAME = "../Files/BetterACpermutation_matrix_250_5.625.mtx"

MODEL_NAME = "Model_8_1_PiOver32_4Layer_DoubleDrop"
 ``` 
 
- Orbit-based:
   - Batch-norm
   - Drop-out
- Other Regularisation:
   - Weight decay : X
   - L1/L2 on weight: X

### Info saved:

```python

experiments/exp_YYYYMMDD_HHMMSS/
├── config.json              # Full config including regularization details
├── training_log.csv         # Loss per epoch
├── learning_curves.png      # Train/val loss plot
├── model_best.pt            # Best checkpoint (includes BN states)
├── model_final.pt           # Final checkpoint
├── val_metrics.json         # Validation metrics (per-output)
├── val_predictions.csv      # Validation predictions
├── val_pred_vs_true.png     # Validation scatter plot
├── val_residuals.png        # Validation residuals
├── test_metrics.json        # Test metrics
├── test_predictions.csv     # Test predictions
├── test_pred_vs_true.png    # Test scatter plot
├── test_residuals.png       # Test residuals
├── equivariance_check.json  # Equivariance verification
└── notes.md                 # Pre-filled notes template

```

In [ ]:
#!/usr/bin/env python3

import os
import glob
import re
import math
import json
import datetime
import csv
from typing import List, Tuple, Sequence
import numpy as np
from scipy.io import mmread
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
from torchvision.io import read_image
from tqdm import tqdm
import matplotlib.pyplot as plt

# ----------------------------
# Global config
# ----------------------------
torch.set_default_dtype(torch.float32)
SEED = 317 #316 # 315 #314
np.random.seed(SEED)
torch.manual_seed(SEED)

# Input Permutation (250x250 image, N=62500)
MTX_INPUT_FILENAME = "../Files/BetterACpermutation_matrix_250_5.625.mtx"

# Layer 1 Permutation (100 dimension)
MTX_LAYER1_FILENAME = "../Files/BetterACpermutation_matrix_10_5.625.mtx"

IMAGE_DIR = "output_ellipses/NewN250/New_Train250x250Images100Ellipses" 
TEST_DIR = "output_ellipses/MoreTestData"


MODEL_NAME = "Model_8.1_4Layer_DoubleDrop"

N_EPOCHS = 10
LR = 1e-3
BATCH_SIZE = 64
TRAIN_SPLIT = 0.8
DROPOUT_RATE = 0.1
TRAINABLE_PARAMS_RATIO = 1.0

NUM_WORKERS = 0 if not torch.cuda.is_available() else 4
PIN_MEMORY = torch.cuda.is_available()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ----------------------------
# Experiment Tracking Setup
# ----------------------------
def create_experiment_folder(base_dir="experiments"):
    """Create a timestamped experiment folder."""
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    exp_name = f"({MODEL_NAME})exp_{timestamp}"
    exp_dir = os.path.join(base_dir, exp_name)
    os.makedirs(exp_dir, exist_ok=True)
    return exp_dir

def save_config(exp_dir: str, config: dict):
    """Save configuration to JSON."""
    config_path = os.path.join(exp_dir, "config.json")
    with open(config_path, 'w') as f:
        json.dump(config, f, indent=2)
    print(f"Saved config to {config_path}")

def save_training_log(exp_dir: str, log: list):
    """Save training log to CSV."""
    if len(log) == 0:
        return
    log_path = os.path.join(exp_dir, "training_log.csv")
    with open(log_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=log[0].keys())
        writer.writeheader()
        writer.writerows(log)
    print(f"Saved training log to {log_path}")

def save_predictions(exp_dir: str, predictions: dict, filename: str):
    """Save predictions to CSV."""
    pred_path = os.path.join(exp_dir, filename)
    with open(pred_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['true_q11', 'true_q12', 'pred_q11', 'pred_q12', 'error_q11', 'error_q12'])
        for i in range(len(predictions['true'])):
            true = predictions['true'][i]
            pred = predictions['pred'][i]
            writer.writerow([
                true[0], true[1],
                pred[0], pred[1],
                abs(true[0] - pred[0]), abs(true[1] - pred[1])
            ])
    print(f"Saved predictions to {pred_path}")

def plot_learning_curves(exp_dir: str, log: list):
    """Plot and save learning curves."""
    if len(log) == 0:
        return
    epochs = [entry['epoch'] for entry in log]
    train_losses = [entry['train_loss'] for entry in log]
    val_losses = [entry['val_loss'] for entry in log]

    plt.figure(figsize=(10, 6))
    plt.plot(epochs, train_losses, 'b-', label='Train Loss', marker='o')
    plt.plot(epochs, val_losses, 'r-', label='Val Loss', marker='s')
    plt.xlabel('Epoch')
    plt.ylabel('MSE Loss')
    plt.title('Learning Curves')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(exp_dir, "learning_curves.png"), dpi=150)
    plt.close()
    print(f"Saved learning curves plot")

def plot_predictions(exp_dir: str, predictions: dict, prefix: str = ""):
    """Plot predicted vs actual for both outputs."""
    true = np.array(predictions['true'])
    pred = np.array(predictions['pred'])

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    for i, label in enumerate(['Q11', 'Q12']):
        ax = axes[i]
        ax.scatter(true[:, i], pred[:, i], alpha=0.5, s=10)
        
        min_val = min(true[:, i].min(), pred[:, i].min())
        max_val = max(true[:, i].max(), pred[:, i].max())
        ax.plot([min_val, max_val], [min_val, max_val], 'r--', label='Perfect')
        
        ax.set_xlabel(f'True {label}')
        ax.set_ylabel(f'Predicted {label}')
        ax.set_title(f'{prefix}{label}: Predicted vs True')
        ax.legend()
        ax.grid(True)

    plt.tight_layout()
    plt.savefig(os.path.join(exp_dir, f"{prefix}pred_vs_true.png"), dpi=150)
    plt.close()
    print(f"Saved {prefix}prediction plot")

def plot_residuals(exp_dir: str, predictions: dict, prefix: str = ""):
    """Plot residual distributions."""
    true = np.array(predictions['true'])
    pred = np.array(predictions['pred'])
    residuals = pred - true

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    for i, label in enumerate(['Q11', 'Q12']):
        ax = axes[i]
        ax.hist(residuals[:, i], bins=50, edgecolor='black', alpha=0.7)
        ax.axvline(x=0, color='r', linestyle='--')
        ax.set_xlabel(f'Residual (Pred - True)')
        ax.set_ylabel('Frequency')
        ax.set_title(f'{prefix}{label} Residuals (mean={residuals[:, i].mean():.4f}, std={residuals[:, i].std():.4f})')
        ax.grid(True)

    plt.tight_layout()
    plt.savefig(os.path.join(exp_dir, f"{prefix}residuals.png"), dpi=150)
    plt.close()
    print(f"Saved {prefix}residual plot")

def compute_metrics(predictions: dict) -> dict:
    """Compute comprehensive metrics."""
    true = np.array(predictions['true'])
    pred = np.array(predictions['pred'])

    metrics = {}

    for i, label in enumerate(['q11', 'q12']):
        t, p = true[:, i], pred[:, i]
        
        mse = np.mean((t - p) ** 2)
        rmse = np.sqrt(mse)
        mae = np.mean(np.abs(t - p))
        
        ss_res = np.sum((t - p) ** 2)
        ss_tot = np.sum((t - np.mean(t)) ** 2)
        r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
        
        max_error = np.max(np.abs(t - p))
        
        metrics[f'{label}_mse'] = float(mse)
        metrics[f'{label}_rmse'] = float(rmse)
        metrics[f'{label}_mae'] = float(mae)
        metrics[f'{label}_r2'] = float(r2)
        metrics[f'{label}_max_error'] = float(max_error)

    combined_mse = np.mean((true - pred) ** 2)
    metrics['combined_mse'] = float(combined_mse)
    metrics['combined_rmse'] = float(np.sqrt(combined_mse))

    return metrics

def save_metrics(exp_dir: str, metrics: dict, filename: str):
    """Save metrics to JSON."""
    metrics_path = os.path.join(exp_dir, filename)
    with open(metrics_path, 'w') as f:
        json.dump(metrics, f, indent=2)
    print(f"Saved metrics to {metrics_path}")

def save_notes(exp_dir: str, notes: str):
    """Save experiment notes."""
    notes_path = os.path.join(exp_dir, "notes.md")
    with open(notes_path, 'w') as f:
        f.write(notes)

# ----------------------------
# Utils: Permutations & Cycles
# ----------------------------
def load_permutation_sigma(filename: str, mock_N=None) -> Tuple[List[int], int]:
    try:
        if not os.path.exists(filename):
            raise FileNotFoundError(f"{filename} not found.")
        G_sparse = mmread(filename).tocsc()
        N = G_sparse.shape[0]
        sigma = [int(G_sparse[:, j].indices[0]) for j in range(N)]
        return sigma, N
    except Exception as e:
        print(f"Warning: {e}.")
        if mock_N is not None:
            print(f"Generating mock permutation for N={mock_N} due to file error.")
            N = mock_N
            # Create a simple shift per rotation or grid cycle
            sigma = list(range(N))
            if N == 250*250:
                # Mock 250 grid
                for i in range(0, N, 250):
                    row = list(range(i, i+250))
                    shifted = row[1:] + row[:1]
                    for k, v in enumerate(shifted):
                        sigma[row[k]] = v
            else:
                # Simple cycle for smaller N
                sigma = [(i + 1) % N for i in range(N)]
            return sigma, N
        else:
            raise e

def cycles_from_sigma(sigma: Sequence[int]) -> List[np.ndarray]:
    N = len(sigma)
    seen = np.zeros(N, dtype=bool)
    cycles = []
    for i in range(N):
        if seen[i]: continue
        cur = i
        cyc = []
        while not seen[cur]:
            cyc.append(cur)
            seen[cur] = True
            cur = sigma[cur]
        cycles.append(np.array(cyc, dtype=np.int64))
    return cycles

def build_orbit_descriptors_from_cycles(left_cycles, right_cycles):
    """
    Build descriptors for Linear map: Right_Vector_Space -> Left_Vector_Space.
    left_cycles: Cycles of the output space.
    right_cycles: Cycles of the input space.
    """
    descriptors = []
    for ia, A in enumerate(left_cycles):
        La = len(A)
        for ib, B in enumerate(right_cycles):
            Lb = len(B)
            g = math.gcd(La, Lb)
            orbit_len = (La * Lb) // g
            for r in range(g):
                descriptors.append((ia, ib, r, orbit_len, La, Lb))
    return descriptors

def expand_descriptor_to_indices(desc, left_cycles, right_cycles):
    ia, ib, r, orbit_len, La, Lb = desc
    A = left_cycles[ia]
    B = right_cycles[ib]
    t = np.arange(orbit_len)
    rows = A[(r + t) % La]
    cols = B[t % Lb]
    return rows, cols

# ----------------------------
# Equivariant Layers
# ----------------------------
class OrbitBatchNorm(nn.Module):
    def __init__(self, cycles: List[np.ndarray], num_features: int, eps=1e-5, momentum=0.1):
        super().__init__()
        self.eps = eps
        self.momentum = momentum
        self.num_cycles = len(cycles)
        
        cycle_indices = torch.zeros(num_features, dtype=torch.long)
        for i, cyc in enumerate(cycles):
            cycle_indices[cyc] = i
        self.register_buffer('cycle_indices', cycle_indices)
        
        cycle_counts = torch.zeros(self.num_cycles, dtype=torch.float32)
        for i, cyc in enumerate(cycles):
            cycle_counts[i] = len(cyc)
        self.register_buffer('cycle_counts', cycle_counts)
        
        self.gamma = nn.Parameter(torch.ones(self.num_cycles))
        self.beta = nn.Parameter(torch.zeros(self.num_cycles))
        
        self.register_buffer('running_mean', torch.zeros(self.num_cycles))
        self.register_buffer('running_var', torch.ones(self.num_cycles))

    def forward(self, x):
        B, N = x.shape
        
        if self.training:
            batch_sum = x.sum(dim=0)
            batch_sq_sum = (x ** 2).sum(dim=0)
            
            orbit_sum = torch.zeros(self.num_cycles, device=x.device)
            orbit_sq_sum = torch.zeros(self.num_cycles, device=x.device)
            
            orbit_sum.scatter_add_(0, self.cycle_indices, batch_sum)
            orbit_sq_sum.scatter_add_(0, self.cycle_indices, batch_sq_sum)
            
            orbit_total_count = self.cycle_counts * B
            batch_mean = orbit_sum / orbit_total_count
            batch_var = (orbit_sq_sum / orbit_total_count) - (batch_mean ** 2)
            
            with torch.no_grad():
                self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * batch_mean
                self.running_var = (1 - self.momentum) * self.running_var + self.momentum * batch_var
            
            mean = batch_mean
            std = torch.sqrt(batch_var + self.eps)
            
        else:
            mean = self.running_mean
            std = torch.sqrt(self.running_var + self.eps)
        
        mean_expanded = mean[self.cycle_indices]
        std_expanded = std[self.cycle_indices]
        gamma_expanded = self.gamma[self.cycle_indices]
        beta_expanded = self.beta[self.cycle_indices]
        
        x_norm = (x - mean_expanded.unsqueeze(0)) / std_expanded.unsqueeze(0)
        return x_norm * gamma_expanded.unsqueeze(0) + beta_expanded.unsqueeze(0)

class OrbitDropout(nn.Module):
    def __init__(self, p: float, cycles: List[np.ndarray], num_features: int):
        super().__init__()
        self.p = p
        self.num_cycles = len(cycles)
        cycle_indices = torch.zeros(num_features, dtype=torch.long)
        for i, cyc in enumerate(cycles):
            cycle_indices[cyc] = i
        self.register_buffer('cycle_indices', cycle_indices)

    def forward(self, x):
        if not self.training or self.p == 0:
            return x
        B = x.shape[0]
        mask_params = torch.full((B, self.num_cycles), 1 - self.p, device=x.device)
        mask_coarse = torch.bernoulli(mask_params)
        mask = mask_coarse[:, self.cycle_indices]
        return x * mask / (1 - self.p)

def make_batched_matvec(rows_flat, cols_flat, pidx_flat, params):
    rows = rows_flat
    cols = cols_flat
    pidx = pidx_flat
    out_dim = int(rows.max().item()) + 1

    def matvec(X):
        X_cols = torch.index_select(X, 1, cols)
        vals = params[pidx].unsqueeze(0)
        weighted = X_cols * vals
        Y = torch.zeros(X.size(0), out_dim, device=X.device, dtype=X.dtype)
        Y.scatter_add_(1, rows.unsqueeze(0).expand(X.size(0), -1), weighted)
        return Y
    return matvec

class Sigma2(nn.Module):
    def __init__(self, L_matrix: torch.Tensor):
        super().__init__()
        self.register_buffer('L', L_matrix)

    def forward(self, y):
        z = y @ self.L.t()
        norm = torch.norm(z, p=2, dim=1, keepdim=True)
        safe_norm = norm + 1e-8
        scale = 0.5 * torch.tanh(norm) / safe_norm
        return z * scale

# ----------------------------
# Dataset
# ----------------------------
class EllipseDataset(Dataset):
    def __init__(self, img_dir: str, preload: bool = True):
        super().__init__()
        self.img_dir = img_dir
        if not os.path.exists(img_dir):
            print(f"Warning: Image directory {img_dir} does not exist.")
            self.img_files = []
        else:
            self.img_files = sorted(glob.glob(os.path.join(img_dir, '*.[pP][nN][gG]')))
            
        self.labels = []
        valid_files = []
        
        if len(self.img_files) == 0:
            print(f"ERROR: No images found in {img_dir}")
            return

        print(f"Found {len(self.img_files)} images. Verifying labels...")
        
        number_pattern = r"([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)"
        pattern = re.compile(rf"Q11[_ ]?{number_pattern}.*?Q12[_ ]?{number_pattern}")
        
        for f in self.img_files:
            filename = os.path.basename(f)
            match = pattern.search(filename)
            if match:
                try:
                    q11_str = match.group(1)
                    q12_str = match.group(2)
                    
                    q11 = float(q11_str)
                    q12 = float(q12_str)
                    
                    self.labels.append(torch.tensor([q11, q12], dtype=torch.float32))
                    valid_files.append(f)
                except ValueError:
                    print(f"Skipping {filename}: Captured '{q11_str}' and '{q12_str}' but could not float convert.")
            else:
                if len(self.labels) < 3: 
                     print(f"Regex mismatch on file: {filename}")
                    
        self.img_files = valid_files
        print(f"Successfully loaded {len(self.labels)} labelled samples.")

        self._preloaded = []
        if preload and len(self.img_files) > 0:
            print(f"Preloading {len(self.img_files)} images...")
            for f in tqdm(self.img_files, desc="Preloading"):
                try:
                    img = read_image(f).float() / 255.0
                    img = transforms.functional.normalize(img, mean=[0.5], std=[0.5])
                    self._preloaded.append(img.flatten())
                except Exception as e:
                    print(f"Error loading {f}: {e}")

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        if self._preloaded:
            return self._preloaded[idx], self.labels[idx]
        img = read_image(self.img_files[idx]).float() / 255.0
        img = transforms.functional.normalize(img, mean=[0.5], std=[0.5])
        return img.flatten(), self.labels[idx]

    def get_label_statistics(self) -> dict:
        if len(self.labels) == 0:
            return {}
        labels = torch.stack(self.labels).numpy()
        return {
            'q11_mean': float(labels[:, 0].mean()),
            'q11_std': float(labels[:, 0].std()),
            'q11_min': float(labels[:, 0].min()),
            'q11_max': float(labels[:, 0].max()),
            'q12_mean': float(labels[:, 1].mean()),
            'q12_std': float(labels[:, 1].std()),
            'q12_min': float(labels[:, 1].min()),
            'q12_max': float(labels[:, 1].max()),
            'num_samples': len(labels)
        }

# ----------------------------
# Model (4 layers)
# ----------------------------
class EquivariantModel(nn.Module):
    def __init__(self, N,
                 # W1 components (250^2 -> 100)
                 rows_w1, cols_w1, pidx_w1, bn1, drop1,
                 # W2 components (100 -> 64)
                 rows_w2, cols_w2, pidx_w2, bn2, drop2,
                 # W3 components (64 -> 32)
                 rows_w3, cols_w3, pidx_w3, bn3,
                 # Output
                 sigma2_layer):
        super().__init__()
        self.N = N

        # Layer 1: G -> G10 (100)
        self.register_buffer('rows_w1', rows_w1)
        self.register_buffer('cols_w1', cols_w1)
        self.register_buffer('pidx_w1', pidx_w1)
        self.bn1 = bn1
        self.drop1 = drop1
        
        # Layer 2: G10 -> Phi64 (64)
        self.register_buffer('rows_w2', rows_w2)
        self.register_buffer('cols_w2', cols_w2)
        self.register_buffer('pidx_w2', pidx_w2)
        self.bn2 = bn2
        self.drop2 = drop2 
        
        # Layer 3: Phi64 -> Phi32 (32)
        self.register_buffer('rows_w3', rows_w3)
        self.register_buffer('cols_w3', cols_w3)
        self.register_buffer('pidx_w3', pidx_w3)
        self.bn3 = bn3
        
        self.sigma2 = sigma2_layer
        
    def forward(self, x, params_w1, params_w2, params_w3):
        # --- Layer 1: R^N -> R^100 (G -> G10 intertwiner) ---
        matvec_1 = make_batched_matvec(self.rows_w1, self.cols_w1, self.pidx_w1, params_w1)
        h = matvec_1(x)
        h = self.bn1(h)
        h = F.gelu(h)  
        h = self.drop1(h) # Dropout Layer 1
        
        # --- Layer 2: R^100 -> R^64 (G10 -> Phi64 intertwiner) ---
        matvec_2 = make_batched_matvec(self.rows_w2, self.cols_w2, self.pidx_w2, params_w2)
        z = matvec_2(h)
        z = self.bn2(z)
        z = F.gelu(z) 
        z = self.drop2(z) # Dropout Layer 2
        
        # --- Layer 3: R^64 -> R^32 (Phi64 -> Phi32 intertwiner) ---
        matvec_3 = make_batched_matvec(self.rows_w3, self.cols_w3, self.pidx_w3, params_w3)
        u = matvec_3(z)
        u = self.bn3(u)
        
        # --- Output: R^32 -> R^2 ---
        out = self.sigma2(u)
        
        return out

# ----------------------------
# Evaluation Function
# ----------------------------
def evaluate_model(model, dataloader, w1_params, w2_params, w3_params, device):
    """Evaluate model and return predictions."""
    model.eval()
    all_true = []
    all_pred = []
    total_loss = 0.0
    loss_fn = nn.MSELoss()

    with torch.no_grad():
        for x_batch, y_batch in dataloader:
            x = x_batch.to(device)
            y = y_batch.to(device)
            pred = model(x, w1_params, w2_params, w3_params)
            total_loss += loss_fn(pred, y).item()
            
            all_true.extend(y.cpu().numpy().tolist())
            all_pred.extend(pred.cpu().numpy().tolist())

    avg_loss = total_loss / len(dataloader) if len(dataloader) > 0 else 0
    return {
        'true': all_true,
        'pred': all_pred,
        'avg_loss': avg_loss
    }

# ----------------------------
# Main
# ----------------------------
def main():
    # Create experiment folder
    exp_dir = create_experiment_folder()
    print(f"Experiment folder: {exp_dir}")

    print(f"Device: {DEVICE}")
    print("Structure: Input(G) -> [W1(100)->BN->GELU->Drop] -> [W2(64)->BN->GELU->Drop] -> [W3(32)->BN] -> Sigma2 -> y")

    # 1. Groups & Cycles
    print(f"Loading INPUT permutation G from {MTX_INPUT_FILENAME}...")
    sigma_perm_G, N = load_permutation_sigma(MTX_INPUT_FILENAME, mock_N=62500)
    print(f"Calculating cycles for Input G (N={N})...")
    cycles_G = cycles_from_sigma(sigma_perm_G)
    print(f"Found {len(cycles_G)} cycles in input.")

    print(f"Loading LAYER 1 permutation G10 from {MTX_LAYER1_FILENAME}...")
    # Assumes the file contains a 100x100 matrix (or similar dim) for the first hidden layer
    sigma_perm_G10, N_G10 = load_permutation_sigma(MTX_LAYER1_FILENAME, mock_N=100)
    print(f"Calculating cycles for G10 (N={N_G10})...")
    cycles_G10 = cycles_from_sigma(sigma_perm_G10)
    print(f"Found {len(cycles_G10)} cycles in G10.")

    # Phi2: 64-cycle (Standard)
    cycles_Phi64 = [np.arange(64, dtype=np.int64)]
    
    # Phi3: 32-cycle (Standard)
    cycles_Phi32 = [np.arange(32, dtype=np.int64)]

    # 2. Build W1 (250^2 -> 100)
    # Mapping G (Input) -> G10 (Output)
    print("Building W1 (G10 x G) descriptors...")
    desc_w1 = build_orbit_descriptors_from_cycles(cycles_G10, cycles_G)
    r1, c1, pi1 = [], [], []
    for i, desc in enumerate(desc_w1):
        r, c = expand_descriptor_to_indices(desc, cycles_G10, cycles_G)
        r1.append(r); c1.append(c); pi1.append(np.full(len(r), i, dtype=np.int64))

    rows_w1 = torch.tensor(np.concatenate(r1), dtype=torch.long, device=DEVICE)
    cols_w1 = torch.tensor(np.concatenate(c1), dtype=torch.long, device=DEVICE)
    pidx_w1 = torch.tensor(np.concatenate(pi1), dtype=torch.long, device=DEVICE)

    w1_params = nn.Parameter(torch.randn(len(desc_w1), device=DEVICE) * 0.01)
    
    # Batch Norm acts on the output of W1 (dimension 100, cycles G10)
    bn1 = OrbitBatchNorm(cycles_G10, 100).to(DEVICE)
    drop1 = OrbitDropout(DROPOUT_RATE, cycles_G10, 100).to(DEVICE)

    # 3. Build W2 (100 -> 64)
    # Mapping G10 -> Phi64
    print("Building W2 (Phi64 x G10) descriptors...")
    desc_w2 = build_orbit_descriptors_from_cycles(cycles_Phi64, cycles_G10)
    r2, c2, pi2 = [], [], []
    for i, desc in enumerate(desc_w2):
        r, c = expand_descriptor_to_indices(desc, cycles_Phi64, cycles_G10)
        r2.append(r); c2.append(c); pi2.append(np.full(len(r), i, dtype=np.int64))
        
    rows_w2 = torch.tensor(np.concatenate(r2), dtype=torch.long, device=DEVICE)
    cols_w2 = torch.tensor(np.concatenate(c2), dtype=torch.long, device=DEVICE)
    pidx_w2 = torch.tensor(np.concatenate(pi2), dtype=torch.long, device=DEVICE)

    w2_params = nn.Parameter(torch.randn(len(desc_w2), device=DEVICE) * 0.01)
    
    # Batch Norm acts on output of W2 (dimension 64, cycles Phi64)
    bn2 = OrbitBatchNorm(cycles_Phi64, 64).to(DEVICE)
    drop2 = OrbitDropout(DROPOUT_RATE, cycles_Phi64, 64).to(DEVICE)

    # 4. Build W3 (64 -> 32)
    # Mapping Phi64 -> Phi32
    print("Building W3 (Phi32 x Phi64) descriptors...")
    desc_w3 = build_orbit_descriptors_from_cycles(cycles_Phi32, cycles_Phi64)
    r3, c3, pi3 = [], [], []
    for i, desc in enumerate(desc_w3):
        r, c = expand_descriptor_to_indices(desc, cycles_Phi32, cycles_Phi64)
        r3.append(r); c3.append(c); pi3.append(np.full(len(r), i, dtype=np.int64))
        
    rows_w3 = torch.tensor(np.concatenate(r3), dtype=torch.long, device=DEVICE)
    cols_w3 = torch.tensor(np.concatenate(c3), dtype=torch.long, device=DEVICE)
    pidx_w3 = torch.tensor(np.concatenate(pi3), dtype=torch.long, device=DEVICE)

    w3_params = nn.Parameter(torch.randn(len(desc_w3), device=DEVICE) * 0.01)
    
    # Batch Norm acts on output of W3 (dimension 32, cycles Phi32)
    bn3 = OrbitBatchNorm(cycles_Phi32, 32).to(DEVICE)

    # 5. Output Projection L (32 -> 2)
    # Constructing symbolic matrix for K=64 (output dim 32)
    # Row 1: cos(k*pi/16), Row 2: sin(k*pi/16) for k in 0..31
    angles = [k * math.pi / 16.0 for k in range(32)]
    row_cos = [math.cos(a) for a in angles]
    row_sin = [math.sin(a) for a in angles]
    
    L_values = [row_cos, row_sin]
    L_matrix = torch.tensor(L_values, dtype=torch.float32, device=DEVICE)
    sigma2_layer = Sigma2(L_matrix).to(DEVICE)

    # 6. Assemble
    model = EquivariantModel(N, 
                             rows_w1, cols_w1, pidx_w1, bn1, drop1,
                             rows_w2, cols_w2, pidx_w2, bn2, drop2, 
                             rows_w3, cols_w3, pidx_w3, bn3,
                             sigma2_layer).to(DEVICE)

    # Count parameters
    num_w1_params = len(w1_params)
    num_w2_params = len(w2_params)
    num_w3_params = len(w3_params)
    num_bn1_params = sum(p.numel() for p in bn1.parameters())
    num_bn2_params = sum(p.numel() for p in bn2.parameters())
    num_bn3_params = sum(p.numel() for p in bn3.parameters())
    total_params = num_w1_params + num_w2_params + num_w3_params + num_bn1_params + num_bn2_params + num_bn3_params

    print(f"\nParameter counts:")
    print(f"  W1: {num_w1_params}, W2: {num_w2_params}, W3: {num_w3_params}")
    print(f"  BN1: {num_bn1_params}, BN2: {num_bn2_params}, BN3: {num_bn3_params}")
    print(f"  Total: {total_params}")

    # 7. Load datasets
    full_dataset = EllipseDataset(img_dir=IMAGE_DIR, preload=True)
    train_data_stats = full_dataset.get_label_statistics()

    test_dataset = None
    test_data_stats = None
    if os.path.exists(TEST_DIR):
        test_dataset = EllipseDataset(img_dir=TEST_DIR, preload=True)
        if len(test_dataset) > 0:
            test_data_stats = test_dataset.get_label_statistics()
            print(f"Loaded test dataset with {len(test_dataset)} samples")
        else:
            test_dataset = None
            print("Test directory exists but no valid samples found")
    else:
        print(f"Test directory not found: {TEST_DIR}")

    # Save configuration
    config = {
        'seed': SEED,
        'device': str(DEVICE),
        'mtx_input_filename': MTX_INPUT_FILENAME,
        'mtx_layer1_filename': MTX_LAYER1_FILENAME,
        'image_dir': IMAGE_DIR,
        'test_dir': TEST_DIR,
        'n_epochs': N_EPOCHS,
        'learning_rate': LR,
        'batch_size': BATCH_SIZE,
        'train_split': TRAIN_SPLIT,
        'dropout_rate': DROPOUT_RATE,
        'trainable_params_ratio': TRAINABLE_PARAMS_RATIO,
        'num_workers': NUM_WORKERS,
        'pin_memory': PIN_MEMORY,
        'model': {
            'architecture': 'Input(G) -> [W1(100)->BN->GELU->Drop] -> [W2(64)->BN->GELU->Drop] -> [W3(32)->BN] -> Sigma2 -> y',
            'num_layers': 4,
            'input_dim': N,
            'hidden1_dim': 100,
            'hidden2_dim': 64,
            'hidden3_dim': 32,
            'output_dim': 2,
            'num_w1_descriptors': num_w1_params,
            'num_w2_descriptors': num_w2_params,
            'num_w3_descriptors': num_w3_params,
            'total_trainable_params': total_params,
            'activations': 'GELU x 2, Sigma2',
            'regularization': f'OrbitDropout (p={DROPOUT_RATE}) on layers 1 AND 2'
        },
        'data': {
            'train_stats': train_data_stats,
            'test_stats': test_data_stats
        }
    }
    save_config(exp_dir, config)

    # 8. Train
    all_trainable = [w1_params, w2_params, w3_params] + \
                    list(bn1.parameters()) + list(bn2.parameters()) + list(bn3.parameters())
    optimizer = optim.Adam(all_trainable, lr=LR)
    loss_function = nn.MSELoss()

    training_log = []
    best_val_loss = float('inf')

    if len(full_dataset) > 0:
        train_size = int(TRAIN_SPLIT * len(full_dataset))
        val_size = len(full_dataset) - train_size
        train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], 
                                                generator=torch.Generator().manual_seed(SEED))
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
                                num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
        
        print(f"\nDataset split: {train_size} train, {val_size} val")
        print("Starting Training...")
        
        for epoch in range(N_EPOCHS):
            model.train()
            train_loss = 0.0
            for x_cpu, y_cpu in tqdm(train_loader, desc=f"Ep {epoch+1}", leave=False):
                x = x_cpu.to(DEVICE, non_blocking=PIN_MEMORY)
                y = y_cpu.to(DEVICE, non_blocking=PIN_MEMORY)
                optimizer.zero_grad()
                pred = model(x, w1_params, w2_params, w3_params)
                loss = loss_function(pred, y)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()
            avg_train = train_loss / len(train_loader)
            
            # Validation
            val_results = evaluate_model(model, val_loader, w1_params, w2_params, w3_params, DEVICE)
            avg_val = val_results['avg_loss']
            
            # Log
            log_entry = {
                'epoch': epoch + 1,
                'train_loss': avg_train,
                'val_loss': avg_val
            }
            training_log.append(log_entry)
            print(f"Epoch {epoch+1}: Train {avg_train:.5f}, Val {avg_val:.5f}")
            
            # Save best model
            if avg_val < best_val_loss:
                best_val_loss = avg_val
                best_checkpoint = {
                    'epoch': epoch + 1,
                    'w1_params': w1_params.detach().cpu(),
                    'w2_params': w2_params.detach().cpu(),
                    'w3_params': w3_params.detach().cpu(),
                    'bn1_state': bn1.state_dict(),
                    'bn2_state': bn2.state_dict(),
                    'bn3_state': bn3.state_dict(),
                    'L_matrix': L_matrix.cpu(),
                    'model_state': model.state_dict(),
                    'optimizer_state': optimizer.state_dict(),
                    'val_loss': avg_val
                }
                torch.save(best_checkpoint, os.path.join(exp_dir, "model_best.pt"))
        
        # Save training log
        save_training_log(exp_dir, training_log)
        plot_learning_curves(exp_dir, training_log)
        
        # Final validation evaluation
        print("\nEvaluating on validation set...")
        val_results = evaluate_model(model, val_loader, w1_params, w2_params, w3_params, DEVICE)
        val_metrics = compute_metrics(val_results)
        save_metrics(exp_dir, val_metrics, "val_metrics.json")
        save_predictions(exp_dir, val_results, "val_predictions.csv")
        plot_predictions(exp_dir, val_results, prefix="val_")
        plot_residuals(exp_dir, val_results, prefix="val_")
        
        print("\nValidation Metrics:")
        for k, v in val_metrics.items():
            print(f"  {k}: {v:.6f}")
            
    else:
        print("Dataset empty. Skipping training.")

    # 9. Test Set Evaluation
    if test_dataset is not None and len(test_dataset) > 0:
        print("\n" + "="*50)
        print("Evaluating on TEST set using BEST model...")
        print("="*50)
        
        best_checkpoint_path = os.path.join(exp_dir, "model_best.pt")
        if os.path.exists(best_checkpoint_path):
            loaded_ckpt = torch.load(best_checkpoint_path, map_location=DEVICE)
            print(f"Loading best model from epoch {loaded_ckpt['epoch']} (val_loss: {loaded_ckpt['val_loss']:.6f})")
            
            w1_params.data = loaded_ckpt['w1_params'].to(DEVICE)
            w2_params.data = loaded_ckpt['w2_params'].to(DEVICE)
            w3_params.data = loaded_ckpt['w3_params'].to(DEVICE)
            bn1.load_state_dict(loaded_ckpt['bn1_state'])
            bn2.load_state_dict(loaded_ckpt['bn2_state'])
            bn3.load_state_dict(loaded_ckpt['bn3_state'])
        else:
            print("WARNING: No best checkpoint found, using final model weights")
        
        test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
        test_results = evaluate_model(model, test_loader, w1_params, w2_params, w3_params, DEVICE)
        test_metrics = compute_metrics(test_results)
        
        save_metrics(exp_dir, test_metrics, "test_metrics.json")
        save_predictions(exp_dir, test_results, "test_predictions.csv")
        plot_predictions(exp_dir, test_results, prefix="test_")
        plot_residuals(exp_dir, test_results, prefix="test_")
        
        print("\nTest Metrics:")
        for k, v in test_metrics.items():
            print(f"  {k}: {v:.6f}")

    # 10. Equivariance Check
    print("\nVerifying Equivariance...")
    sigma_inv = [0]*N
    for i, s in enumerate(sigma_perm_G): sigma_inv[s] = i
    sigma_idx = torch.tensor(sigma_inv, device=DEVICE, dtype=torch.long)
    
    # Logic:
    # Input permutation G corresponds to theta = 2*pi/64 (approx 5.625 deg).
    # The output (Q11, Q12) rotates by 2*theta.
    # Therefore, Output Rotation angle = 2 * (2*pi/64) = pi/16.
    
    model.eval()
    with torch.no_grad():
        x = torch.randn(1, N, device=DEVICE)
        x = (x - 0.5) / 0.5 
        
        out_x = model(x, w1_params, w2_params, w3_params).squeeze(0)
        
        # Apply G to input (Simulating 5.625 deg input rotation)
        x_G = torch.index_select(x, 1, sigma_idx)
        out_Gx = model(x_G, w1_params, w2_params, w3_params).squeeze(0)
        
        # Apply R(pi/16) to output (Simulating 2 * 5.625 = 11.25 deg output rotation)
        ang = math.pi / 16.0
        R_out = torch.tensor([
            [math.cos(ang), -math.sin(ang)], 
            [math.sin(ang), math.cos(ang)]
        ], device=DEVICE)
        
        out_expected = R_out @ out_x
        
        diff = (out_Gx - out_expected).norm().item()
        
        print(f"Output vector f(x):        {out_x.cpu().numpy()}")
        print(f"Model f(G*x) [Input 5.625]:{out_Gx.cpu().numpy()}")
        print(f"Expected R(11.25)*f(x):    {out_expected.cpu().numpy()}")
        print(f"Equivariance Error (Diff): {diff:.6e}")
        
        equivariance_check = {
            'output_vector': out_x.cpu().numpy().tolist(),
            'model_Gx': out_Gx.cpu().numpy().tolist(),
            'diff_raw': diff,
        }

    with open(os.path.join(exp_dir, "equivariance_check.json"), 'w') as f:
        json.dump(equivariance_check, f, indent=2)

    # 11. Save final checkpoint
    final_checkpoint = {
        'epoch': N_EPOCHS,
        'w1_params': w1_params.detach().cpu(),
        'w2_params': w2_params.detach().cpu(),
        'w3_params': w3_params.detach().cpu(),
        'bn1_state': bn1.state_dict(),
        'bn2_state': bn2.state_dict(),
        'bn3_state': bn3.state_dict(),
        'L_matrix': L_matrix.cpu(),
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'desc': '4-Layer Arch Pi/32: W1->BN->GELU->Drop -> W2->BN->GELU->Drop -> W3->BN -> Sigma2 -> y'
    }
    torch.save(final_checkpoint, os.path.join(exp_dir, "model_final.pt"))

    # 12. Save experiment notes template
    notes = f"""# Experiment Notes
Date: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

Architecture
Input(250^2) -> W1(100) -> BN -> GELU -> Drop -> W2(64) -> BN -> GELU -> Drop -> W3(32) -> BN -> Sigma2 -> Output(2)

Key Hyperparameters
Learning Rate: {LR}
Batch Size: {BATCH_SIZE}
Epochs: {N_EPOCHS}
Dropout Rate: {DROPOUT_RATE} (Applied twice)
Total Parameters: {total_params}

Regularization
OrbitBatchNorm: Per-cycle batch normalization WITH HISTORICAL STATS
OrbitDropout: Drops entire cycles (p={DROPOUT_RATE}) on layers 1 AND 2

Key Results
Best validation loss: {best_val_loss:.6f}
Equivariance diff (vs Rot 11.25 deg): {diff:.6e}
"""
    save_notes(exp_dir, notes)

    print("\n" + "="*50)
    print("EXPERIMENT SUMMARY")
    print("="*50)
    print(f"Experiment saved to: {exp_dir}")
    print(f"\nFiles saved:")
    for f in sorted(os.listdir(exp_dir)):
        size = os.path.getsize(os.path.join(exp_dir, f))
        print(f"  {f} ({size:,} bytes)")

    print(f"Total trainable parameters: {total_params}")
    print(f"Best validation loss: {best_val_loss:.6f}")

if __name__ == '__main__':
    main()